# Manufacturing Agent — RAG 전용 시나리오 노트북

`scripts/run_rag_scenarios.py` 의 RAG(문서 근거) 시나리오를 **시나리오 1개당 셀 1개**로 분리한 실행형 노트북이다. 각 셀에는 `Scenario(...)` 정의가 펼쳐져 있고, 실행하면 **소요 시간 + 유사도 score + rag_trace(fan-out/priority/source/score)** 를 출력한다.

사용법:
1. 아래 **부트스트랩 셀**을 먼저 1회 실행한다 (런타임 `g` 로드, LLM 포함).
2. 원하는 시나리오 셀을 실행한다. 셀 안의 `Scenario(...)` 를 직접 고쳐 실험해도 된다.
3. 맨 아래 **요약 셀**로 PASS/FAIL + 소요 시간을 집계한다.

검증 기준(check)은 `run_rag_scenarios.py` 의 것을 재사용한다(`R._check_*`).
이 노트북은 `scripts/_gen_rag_scenarios_nb.py` 로 자동 생성된다.

In [1]:
# === 부트스트랩 (제일 먼저 1회 실행) =========================================
# RAG 전용 러너(run_rag_scenarios)를 라이브러리로 재사용한다.
# 노트북 런타임(g)을 1회 로드해 모든 시나리오 셀이 공유한다.
import os, sys, time, uuid
from pathlib import Path

# 이 노트북은 scripts/ 에 있다. 어디서 열든 작업 디렉터리를 저장소 루트로 고정해야
# 메인 노트북의 상대경로(agent_data/...: DB·chroma)가 깨지지 않는다.
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "scripts"))
print("작업 디렉터리:", Path.cwd())

# (선택) retrieval layer 디버그 로그를 stderr로 보고 싶으면 주석 해제
# os.environ["RAG_DEBUG"] = "true"

import run_manufacturing_scenarios_v2 as v2     # 보정된 DEFINITION_CELLS 로더 재사용
import run_rag_scenarios as R                    # RAG 전용 check / 출력 헬퍼
from run_manufacturing_scenarios import Turn, Scenario, FEATURES_HIGH_RISK

g = v2._load_runtime()                           # manufacturing_agent_v6.ipynb 런타임(LLM 포함)
RESULTS: dict[str, bool] = {}                    # 셀별 PASS/FAIL 누적
TIMINGS: dict[str, float] = {}                   # 셀별 소요 시간(초) 누적


def run_rag(sc, full_answer: bool = True, trace: bool = True):
    """시나리오 1개 실행 — 소요 시간 + 유사도 score + rag_trace 출력."""
    rid = f"rag-{int(time.time())}-{uuid.uuid4().hex[:8]}"   # 매 실행 고유 thread (체크포인트 충돌 방지)
    return R.run_rag_inline(g, sc, rid, RESULTS, TIMINGS, full_answer=full_answer, trace=trace)


print("부트스트랩 완료. RAG 시나리오 수:", len(R.scenarios()))


작업 디렉터리: c:\Users\user\Desktop\sesac-manufacturing-1st-project
LangGraph import 완료
SqliteSaver 사용 가능: True | ToolNode 사용 가능: True
.env file: OK
.env loaded: OK
OpenAI API key: OK
Chat model: gpt-4o
Embedding model: text-embedding-3-small
Use OpenAI embeddings: YES
LangSmith tracing: OFF
LangSmith API key: MISSING
LangSmith project: manufacturing-agent
LangSmith endpoint: https://api.smith.langchain.com
LangSmith upload check: SKIPPED
LLM 모드: REAL(default=gpt-4o, classifier=gpt-4o-mini, final=gpt-4o/2048)
contracts 정의 완료 (SupervisorPlan Orchestrator + typed artifacts)
ManufacturingState(MessagesState 상속) 정의 완료
장기 메모리(SQLite) 준비 완료: agent_data\longterm_memory.sqlite
Evidence RAG ChromaDB 연결 완료: collection=manufacturing_document_chunks_openai, embedding=OpenAI(text-embedding-3-small), chunks=827
Evidence RAG vector_search 준비 완료
context_policy 정의 완료
context_selector 정의 완료
context_normalizer 정의 완료
context_packer 정의 완료
prediction_service 정의 완료
LangGraph import 완료
SqliteSaver 사용 가능: True | To

## 원인 (cause) — 단순 RAG

### `RAG_CAUSE_01` — 스핀들이 과열되는 원인은 뭐야?

**대화 턴:**
1. 스핀들이 과열되는 원인은 뭐야?

check: `_check_rag_only` | 태그: cause, rag

In [2]:
sc = Scenario(
    sid='RAG_CAUSE_01',
    description='스핀들이 과열되는 원인은 뭐야?',
    turns=[
        Turn('스핀들이 과열되는 원인은 뭐야?'),
    ],
    check=R._check_rag_only,
    tags=['cause', 'rag'],
)
summary = run_rag(sc)

[RAG_CAUSE_01] 스핀들이 과열되는 원인은 뭐야?
  PASS ✅ (12.58s)
  tasks=['evidence', 'final_answer']
  유사도 score: top=0.586 평균=0.567 전체=[0.586, 0.575, 0.563, 0.542]
    rag_trace: status=OK profile=troubleshooting_rag mode=A
      search_query: 스핀들이 과열되는 원인은 뭐야?

[Supervisor evidence focus]
과열 원인, 스핀들 overheating overheat spindle temperature temperature at the top of the spindle taper getting too hot heat temperature thermal expansion
      expansion_tags: ['overheating', 'overheat', 'spindle temperature', 'temperature at the top of the spindle taper', 'getting too hot', 'heat', 'temperature', 'thermal expansion', 'thermal growth', '과열', '발열', '온도 상승']
      priority_docs: ['Mill Spindle'] | failure_types: []
      retrieved (source/chunk/score):
        - haas/haascnc.com-Mill Spindle - Troubleshooting Guide.pdf (chunk=7, page=None) score=0.586
        - haas/haascnc.com-Mill Spindle - Troubleshooting Guide.pdf (chunk=6, page=None) score=0.575
        - haas/haascnc.com-Mill Spindle - Troubleshoot

### `RAG_CAUSE_02` — 밀링 머신 진동이 심한 이유가 뭐야?

**대화 턴:**
1. 밀링 머신 진동이 심한 이유가 뭐야?

check: `_check_rag_only` | 태그: cause, rag

In [4]:
sc = Scenario(
    sid='RAG_CAUSE_02',
    description='밀링 머신 진동이 심한 이유가 뭐야?',
    turns=[
        Turn('밀링 머신 진동이 심한 이유가 뭐야?'),
    ],
    check=R._check_rag_only,
    tags=['cause', 'rag'],
)
summary = run_rag(sc)

[RAG_CAUSE_02] 밀링 머신 진동이 심한 이유가 뭐야?
  PASS ✅ (12.21s)
  tasks=['evidence', 'final_answer']
  유사도 score: top=0.666 평균=0.628 전체=[0.666, 0.641, 0.612, 0.593]
    rag_trace: status=OK profile=troubleshooting_rag mode=A
      search_query: 밀링 머신 진동이 심한 이유가 뭐야?

[Supervisor evidence focus]
진동 원인, 밀링 머신, 기계 진동 chatter chattered surface finish surface finish shows chatter spindle vibrates vibration excessive vibration resonate poor finish
      expansion_tags: ['chatter', 'chattered surface finish', 'surface finish shows chatter', 'spindle vibrates', 'vibration', 'excessive vibration', 'resonate', 'poor finish', 'finish issue', '채터', '진동', '떨림']
      priority_docs: ['Mill Chatter'] | failure_types: []
      retrieved (source/chunk/score):
        - haas/english---mechanical-service-manual---2007.pdf (chunk=4, page=None) score=0.666
        - haas/haascnc.com-Mill Chatter - Troubleshooting - TG0100.pdf (chunk=2, page=None) score=0.641
        - haas/haascnc.com-Mill Spindle - Troubleshooting G

### `RAG_CAUSE_03` — 절삭면 품질이 갑자기 나빠졌는데 왜 그럴까?

**대화 턴:**
1. 절삭면 품질이 갑자기 나빠졌는데 왜 그럴까?

check: `_check_rag_only` | 태그: cause, rag

In [5]:
sc = Scenario(
    sid='RAG_CAUSE_03',
    description='절삭면 품질이 갑자기 나빠졌는데 왜 그럴까?',
    turns=[
        Turn('절삭면 품질이 갑자기 나빠졌는데 왜 그럴까?'),
    ],
    check=R._check_rag_only,
    tags=['cause', 'rag'],
)
summary = run_rag(sc)

[RAG_CAUSE_03] 절삭면 품질이 갑자기 나빠졌는데 왜 그럴까?
  PASS ✅ (10.14s)
  tasks=['evidence', 'final_answer']
  유사도 score: (검색 문서 없음 / NO_EVIDENCE)
    rag_trace: status=EMPTY profile=fallback_broad mode=A
      search_query: 절삭면 품질이 갑자기 나빠졌는데 왜 그럴까?

[Supervisor evidence focus]
절삭면 품질 저하 원인, 절삭 공정 문제 해결 방법, failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요., failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.

[Gate feedback]
failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.
      expansion_tags: []
      priority_docs: ['Mechanical Service', 'Mill Spindle', 'Mill Chatter'] | failure_types: []
      retrieved: (없음)
  answer=📄 종합 판단: 점검 문서 근거 요약  절삭면 품질 저하는 공구 마모, 절삭 조건 변화, 또는 기계 정렬 문제일 가능성이 있으니 점검이 필요합니다.  문서 근거 현재 검색된 문서 근거만으로는 단정하기 어렵습니다.  ℹ 보조 진단·조회 결과이며, 실제 조치는 현장 담당자 확인이 필요합니다.
  --- 최종 답변 ---
    📄 종합 판단: 점검 문서 근거 요약
    
    절삭면 품질 저하는 

### `RAG_CAUSE_04` — 공구가 빨리 마모되는 원인은?

**대화 턴:**
1. 공구가 빨리 마모되는 원인은?

check: `_check_rag_only` | 태그: cause, rag

In [6]:
sc = Scenario(
    sid='RAG_CAUSE_04',
    description='공구가 빨리 마모되는 원인은?',
    turns=[
        Turn('공구가 빨리 마모되는 원인은?'),
    ],
    check=R._check_rag_only,
    tags=['cause', 'rag'],
)
summary = run_rag(sc)

[RAG_CAUSE_04] 공구가 빨리 마모되는 원인은?
  PASS ✅ (13.33s)
  tasks=['evidence', 'final_answer']
  유사도 score: top=0.459 평균=0.459 전체=[0.459]
    rag_trace: status=OK profile=fallback_broad mode=A
      search_query: 공구가 빨리 마모되는 원인은?

[Supervisor evidence focus]
공구 마모 원인, 마모 방지 방법, failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요., failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.

[Gate feedback]
failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.
      expansion_tags: []
      priority_docs: ['Mechanical Service', 'Mill Spindle', 'Mill Chatter'] | failure_types: []
      retrieved (source/chunk/score):
        - kosha/M-131-2012 기계의 결함진단을 위한 자료해석 기술지침.pdf (chunk=1, page=None) score=0.459
  answer=📄 종합 판단: 점검 문서 근거 요약  공구의 빠른 마모 원인은 공구 재질, 절삭 조건, 냉각제 사용 여부 등 다양한 요인이 있을 수 있으므로 관련 문서를 참고하여 점검이 필요합니다.  문서 근거 - 공구 마모의 주요 원인은 기계의 비정상적인 상태나 거동, 

### `RAG_CAUSE_05` — 토크가 높아지면 어떤 문제가 생겨?

**대화 턴:**
1. 토크가 높아지면 어떤 문제가 생겨?

check: `_check_rag_only` | 태그: cause, rag

In [7]:
sc = Scenario(
    sid='RAG_CAUSE_05',
    description='토크가 높아지면 어떤 문제가 생겨?',
    turns=[
        Turn('토크가 높아지면 어떤 문제가 생겨?'),
    ],
    check=R._check_rag_only,
    tags=['cause', 'rag'],
)
summary = run_rag(sc)

[RAG_CAUSE_05] 토크가 높아지면 어떤 문제가 생겨?
  PASS ✅ (13.56s)
  tasks=['evidence', 'final_answer']
  유사도 score: top=0.454 평균=0.454 전체=[0.454]
    rag_trace: status=OK profile=fallback_broad mode=A
      search_query: 토크가 높아지면 어떤 문제가 생겨?

[Supervisor evidence focus]
토크 증가로 인한 문제 설명, 안전 절차, 장비 손상 원인, failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.

[Gate feedback]
failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.
      expansion_tags: []
      priority_docs: ['Mechanical Service', 'Mill Spindle', 'Mill Chatter'] | failure_types: []
      retrieved (source/chunk/score):
        - kosha/M-192-2017 기계안전을 위한 제어시스템의 안전관련부품류 설계 기술지침.pdf (chunk=33, page=None) score=0.454
  answer=📄 종합 판단: 점검 문서 근거 요약  토크가 높아지면 기계 부품의 과부하로 인해 손상이나 마모가 발생할 수 있으므로, 관련 문서를 참조하여 기계 상태를 점검할 필요가 있다.  문서 근거 - 토크가 높아지면 기계의 안전기능이 손실될 가능성이 증가할 수 있다. 이는 안전기능의 결함이 발생할 경우, 기계가 안전상태를 유지하지 못하고 위험한 상황을 초래할 수 있기 때문이다[C1]. - 안전기능의

### `RAG_CAUSE_06` — 회전수가 너무 낮으면 어떤 영향이 있어?

**대화 턴:**
1. 회전수가 너무 낮으면 어떤 영향이 있어?

check: `_check_rag_only` | 태그: cause, rag

In [8]:
sc = Scenario(
    sid='RAG_CAUSE_06',
    description='회전수가 너무 낮으면 어떤 영향이 있어?',
    turns=[
        Turn('회전수가 너무 낮으면 어떤 영향이 있어?'),
    ],
    check=R._check_rag_only,
    tags=['cause', 'rag'],
)
summary = run_rag(sc)

[RAG_CAUSE_06] 회전수가 너무 낮으면 어떤 영향이 있어?
  PASS ✅ (16.10s)
  tasks=['evidence', 'final_answer']
  유사도 score: top=0.477 평균=0.472 전체=[0.477, 0.475, 0.47, 0.468]
    rag_trace: status=OK profile=fallback_broad mode=A
      search_query: 회전수가 너무 낮으면 어떤 영향이 있어?

[Supervisor evidence focus]
회전수가 낮을 때의 영향, 관련 안전 절차, 장비 성능

[Gate feedback]
failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.
      expansion_tags: []
      priority_docs: ['Mechanical Service', 'Mill Spindle', 'Mill Chatter'] | failure_types: []
      retrieved (source/chunk/score):
        - kosha/M-192-2017 기계안전을 위한 제어시스템의 안전관련부품류 설계 기술지침.pdf (chunk=28, page=None) score=0.477
        - kosha/B-M-25-2026 에너지 차단장치의 잠금·표지에 관한 기술지원규정.pdf (chunk=1, page=None) score=0.475
        - kosha/M-121-2012 성능매개변수를 이용한 기계의 상태감시와 진단 기술지침.pdf (chunk=4, page=None) score=0.47
        - kosha/M-192-2017 기계안전을 위한 제어시스템의 안전관련부품류 설계 기술지침.pdf (chunk=26, page=None) score=0.468
  answer=📄 종합 판단: 점검 문서 근거 

### `RAG_CAUSE_07` — 공정 온도가 높으면 어떤 문제가 발생해?

**대화 턴:**
1. 공정 온도가 높으면 어떤 문제가 발생해?

check: `_check_rag_only` | 태그: cause, rag

In [9]:
sc = Scenario(
    sid='RAG_CAUSE_07',
    description='공정 온도가 높으면 어떤 문제가 발생해?',
    turns=[
        Turn('공정 온도가 높으면 어떤 문제가 발생해?'),
    ],
    check=R._check_rag_only,
    tags=['cause', 'rag'],
)
summary = run_rag(sc)

[RAG_CAUSE_07] 공정 온도가 높으면 어떤 문제가 발생해?
  PASS ✅ (12.14s)
  tasks=['evidence', 'final_answer']
  유사도 score: top=0.586 평균=0.567 전체=[0.586, 0.575, 0.563, 0.542]
    rag_trace: status=OK profile=troubleshooting_rag mode=A
      search_query: 공정 온도가 높으면 어떤 문제가 발생해?

[Supervisor evidence focus]
온도 상승에 따른 문제점, 안전 절차, 공정 영향 overheating overheat spindle temperature temperature at the top of the spindle taper getting too hot heat temperature thermal expansion
      expansion_tags: ['overheating', 'overheat', 'spindle temperature', 'temperature at the top of the spindle taper', 'getting too hot', 'heat', 'temperature', 'thermal expansion', 'thermal growth', '과열', '발열', '온도 상승']
      priority_docs: ['Mill Spindle'] | failure_types: []
      retrieved (source/chunk/score):
        - haas/haascnc.com-Mill Spindle - Troubleshooting Guide.pdf (chunk=7, page=None) score=0.586
        - haas/haascnc.com-Mill Spindle - Troubleshooting Guide.pdf (chunk=6, page=None) score=0.575
        - haas/haascnc.com-

### `RAG_CAUSE_08` — 냉각이 제대로 안 되면 어떤 고장이 발생할 수 있어?

**대화 턴:**
1. 냉각이 제대로 안 되면 어떤 고장이 발생할 수 있어?

check: `_check_rag_only` | 태그: cause, rag

In [10]:
sc = Scenario(
    sid='RAG_CAUSE_08',
    description='냉각이 제대로 안 되면 어떤 고장이 발생할 수 있어?',
    turns=[
        Turn('냉각이 제대로 안 되면 어떤 고장이 발생할 수 있어?'),
    ],
    check=R._check_rag_only,
    tags=['cause', 'rag'],
)
summary = run_rag(sc)

[RAG_CAUSE_08] 냉각이 제대로 안 되면 어떤 고장이 발생할 수 있어?
  PASS ✅ (10.13s)
  tasks=['evidence', 'final_answer']
  유사도 score: (검색 문서 없음 / NO_EVIDENCE)
    rag_trace: status=EMPTY profile=fallback_broad mode=A
      search_query: 냉각이 제대로 안 되면 어떤 고장이 발생할 수 있어?

[Supervisor evidence focus]
냉각 시스템 고장 원인, 냉각 실패 시 고장 유형, 냉각 관련 안전 절차, failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요., failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.

[Gate feedback]
failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.
      expansion_tags: []
      priority_docs: ['Mechanical Service', 'Mill Spindle', 'Mill Chatter'] | failure_types: []
      retrieved: (없음)
  answer=📄 종합 판단: 점검 문서 근거 요약  냉각이 제대로 되지 않으면 장비 과열로 인해 성능 저하, 부품 손상, 시스템 오류 등이 발생할 수 있으니 냉각 시스템 점검이 필요합니다.  문서 근거 현재 검색된 문서 근거만으로는 단정하기 어렵습니다.  ℹ 보조 진단·조회 결과이며, 실제 조치는 현장 담당자 확인이 필요합니다.
  --- 최종 답변 ---
    📄 종

## 점검 (inspection) — 단순 RAG

### `RAG_INSPECTION_01` — 스핀들 점검 절차 알려줘.

**대화 턴:**
1. 스핀들 점검 절차 알려줘.

check: `_check_rag_only` | 태그: inspection, rag

In [11]:
sc = Scenario(
    sid='RAG_INSPECTION_01',
    description='스핀들 점검 절차 알려줘.',
    turns=[
        Turn('스핀들 점검 절차 알려줘.'),
    ],
    check=R._check_rag_only,
    tags=['inspection', 'rag'],
)
summary = run_rag(sc)

[RAG_INSPECTION_01] 스핀들 점검 절차 알려줘.
  PASS ✅ (9.18s)
  tasks=['evidence', 'final_answer']
  유사도 score: (검색 문서 없음 / NO_EVIDENCE)
    rag_trace: status=EMPTY profile=fallback_broad mode=A
      search_query: 스핀들 점검 절차 알려줘.

[Supervisor evidence focus]
스핀들 점검 절차, failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요., failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.

[Gate feedback]
failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.
      expansion_tags: []
      priority_docs: ['Mechanical Service', 'Mill Spindle', 'Mill Chatter'] | failure_types: []
      retrieved: (없음)
  answer=📄 종합 판단: 점검 문서 근거 요약  스핀들 점검을 위해 먼저 장비의 전원을 차단하고, 스핀들의 외관 및 연결 상태를 육안으로 확인한 후, 이상 소음이나 진동이 있는지 점검해야 합니다.  문서 근거 현재 검색된 문서 근거만으로는 단정하기 어렵습니다.  ℹ 보조 진단·조회 결과이며, 실제 조치는 현장 담당자 확인이 필요합니다.
  --- 최종 답변 ---
    📄 종합 판단: 점검 문서 근거 요약
    
    스핀들 점검을 위해 먼저 장비의 전원을 차단

### `RAG_INSPECTION_02` — 공구 마모는 어떻게 확인해?

**대화 턴:**
1. 공구 마모는 어떻게 확인해?

check: `_check_rag_only` | 태그: inspection, rag

In [12]:
sc = Scenario(
    sid='RAG_INSPECTION_02',
    description='공구 마모는 어떻게 확인해?',
    turns=[
        Turn('공구 마모는 어떻게 확인해?'),
    ],
    check=R._check_rag_only,
    tags=['inspection', 'rag'],
)
summary = run_rag(sc)

[RAG_INSPECTION_02] 공구 마모는 어떻게 확인해?
  PASS ✅ (8.75s)
  tasks=['evidence', 'final_answer']
  유사도 score: (검색 문서 없음 / NO_EVIDENCE)
    rag_trace: status=EMPTY profile=fallback_broad mode=A
      search_query: 공구 마모는 어떻게 확인해?

[Supervisor evidence focus]
공구 마모 확인 방법, failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요., failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.

[Gate feedback]
failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.
      expansion_tags: []
      priority_docs: ['Mechanical Service', 'Mill Spindle', 'Mill Chatter'] | failure_types: []
      retrieved: (없음)
  answer=📄 종합 판단: 점검 문서 근거 요약  공구 마모는 정기적인 시각적 검사와 측정 장비를 사용한 정밀 측정을 통해 확인이 필요하다.  문서 근거 현재 검색된 문서 근거만으로는 단정하기 어렵습니다.  ℹ 보조 진단·조회 결과이며, 실제 조치는 현장 담당자 확인이 필요합니다.
  --- 최종 답변 ---
    📄 종합 판단: 점검 문서 근거 요약
    
    공구 마모는 정기적인 시각적 검사와 측정 장비를 사용한 정밀 측정을 통해 확인이 필요하다.
 

### `RAG_INSPECTION_03` — 벨트 이상 여부는 어떻게 점검해?

**대화 턴:**
1. 벨트 이상 여부는 어떻게 점검해?

check: `_check_rag_only` | 태그: inspection, rag

In [ ]:
sc = Scenario(
    sid='RAG_INSPECTION_03',
    description='벨트 이상 여부는 어떻게 점검해?',
    turns=[
        Turn('벨트 이상 여부는 어떻게 점검해?'),
    ],
    check=R._check_rag_only,
    tags=['inspection', 'rag'],
)
summary = run_rag(sc)

### `RAG_INSPECTION_04` — 냉각 시스템 점검 순서 알려줘.

**대화 턴:**
1. 냉각 시스템 점검 순서 알려줘.

check: `_check_rag_only` | 태그: inspection, rag

In [13]:
sc = Scenario(
    sid='RAG_INSPECTION_04',
    description='냉각 시스템 점검 순서 알려줘.',
    turns=[
        Turn('냉각 시스템 점검 순서 알려줘.'),
    ],
    check=R._check_rag_only,
    tags=['inspection', 'rag'],
)
summary = run_rag(sc)

[RAG_INSPECTION_04] 냉각 시스템 점검 순서 알려줘.
  PASS ✅ (10.92s)
  tasks=['evidence', 'final_answer']
  유사도 score: top=0.453 평균=0.453 전체=[0.453]
    rag_trace: status=OK profile=fallback_broad mode=A
      search_query: 냉각 시스템 점검 순서 알려줘.

[Supervisor evidence focus]
냉각 시스템 점검 순서, failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.

[Gate feedback]
failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.
      expansion_tags: []
      priority_docs: ['Mechanical Service', 'Mill Spindle', 'Mill Chatter'] | failure_types: []
      retrieved (source/chunk/score):
        - kosha/M-192-2017 기계안전을 위한 제어시스템의 안전관련부품류 설계 기술지침.pdf (chunk=33, page=None) score=0.453
  answer=📄 종합 판단: 점검 문서 근거 요약  냉각 시스템의 점검은 관련 문서를 참조하여 순서와 절차를 확인한 후 진행해야 합니다.  문서 근거 - 냉각 시스템의 안전기능 점검은 기계 시동 시, 위험한 상황의 개시 전, 새로운 주기의 시작 또는 다른 동작의 시작 시 주기적으로 수행되어야 한다. 이는 시스템의 안전성을 유지하기 위한 필수적인 절차이다[C1]. - 안전기능의 점검은 자동으로 시작될 수 있으며, 결함이 검출되지 않았다면

### `RAG_INSPECTION_05` — 베어링 이상은 어떻게 확인하지?

**대화 턴:**
1. 베어링 이상은 어떻게 확인하지?

check: `_check_rag_only` | 태그: inspection, rag

In [14]:
sc = Scenario(
    sid='RAG_INSPECTION_05',
    description='베어링 이상은 어떻게 확인하지?',
    turns=[
        Turn('베어링 이상은 어떻게 확인하지?'),
    ],
    check=R._check_rag_only,
    tags=['inspection', 'rag'],
)
summary = run_rag(sc)

[RAG_INSPECTION_05] 베어링 이상은 어떻게 확인하지?
  PASS ✅ (12.12s)
  tasks=['evidence', 'final_answer']
  유사도 score: (검색 문서 없음 / NO_EVIDENCE)
    rag_trace: status=EMPTY profile=fallback_broad mode=A
      search_query: 베어링 이상은 어떻게 확인하지?

[Supervisor evidence focus]
베어링 이상 확인 방법, 진단 절차, 점검 방법, failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요., failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.

[Gate feedback]
failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.
      expansion_tags: []
      priority_docs: ['Mechanical Service', 'Mill Spindle', 'Mill Chatter'] | failure_types: []
      retrieved: (없음)
  answer=📄 종합 판단: 점검 문서 근거 요약  베어링 이상 여부는 소음, 진동, 온도 변화를 통해 확인이 필요하며, 정밀 점검을 위해 전문 장비를 사용하는 것이 좋습니다.  문서 근거 현재 검색된 문서 근거만으로는 단정하기 어렵습니다.  ℹ 보조 진단·조회 결과이며, 실제 조치는 현장 담당자 확인이 필요합니다.
  --- 최종 답변 ---
    📄 종합 판단: 점검 문서 근거 요약
    
    베어링 이상 여부는 소음

### `RAG_INSPECTION_06` — 진동이 심할 때 가장 먼저 확인할 부분은?

**대화 턴:**
1. 진동이 심할 때 가장 먼저 확인할 부분은?

check: `_check_rag_only` | 태그: inspection, rag

In [ ]:
sc = Scenario(
    sid='RAG_INSPECTION_06',
    description='진동이 심할 때 가장 먼저 확인할 부분은?',
    turns=[
        Turn('진동이 심할 때 가장 먼저 확인할 부분은?'),
    ],
    check=R._check_rag_only,
    tags=['inspection', 'rag'],
)
summary = run_rag(sc)

### `RAG_INSPECTION_07` — 모터 이상 여부를 확인하는 방법 알려줘.

**대화 턴:**
1. 모터 이상 여부를 확인하는 방법 알려줘.

check: `_check_rag_only` | 태그: inspection, rag

In [ ]:
sc = Scenario(
    sid='RAG_INSPECTION_07',
    description='모터 이상 여부를 확인하는 방법 알려줘.',
    turns=[
        Turn('모터 이상 여부를 확인하는 방법 알려줘.'),
    ],
    check=R._check_rag_only,
    tags=['inspection', 'rag'],
)
summary = run_rag(sc)

## 정비 (maintenance) — 단순 RAG

### `RAG_MAINTENANCE_01` — 공구를 언제 교체해야 해?

**대화 턴:**
1. 공구를 언제 교체해야 해?

check: `_check_rag_only` | 태그: maintenance, rag

In [ ]:
sc = Scenario(
    sid='RAG_MAINTENANCE_01',
    description='공구를 언제 교체해야 해?',
    turns=[
        Turn('공구를 언제 교체해야 해?'),
    ],
    check=R._check_rag_only,
    tags=['maintenance', 'rag'],
)
summary = run_rag(sc)

### `RAG_MAINTENANCE_02` — 스핀들 정비 절차 알려줘.

**대화 턴:**
1. 스핀들 정비 절차 알려줘.

check: `_check_rag_only` | 태그: maintenance, rag

In [ ]:
sc = Scenario(
    sid='RAG_MAINTENANCE_02',
    description='스핀들 정비 절차 알려줘.',
    turns=[
        Turn('스핀들 정비 절차 알려줘.'),
    ],
    check=R._check_rag_only,
    tags=['maintenance', 'rag'],
)
summary = run_rag(sc)

### `RAG_MAINTENANCE_03` — 냉각장치 청소는 어떻게 해?

**대화 턴:**
1. 냉각장치 청소는 어떻게 해?

check: `_check_rag_only` | 태그: maintenance, rag

In [ ]:
sc = Scenario(
    sid='RAG_MAINTENANCE_03',
    description='냉각장치 청소는 어떻게 해?',
    turns=[
        Turn('냉각장치 청소는 어떻게 해?'),
    ],
    check=R._check_rag_only,
    tags=['maintenance', 'rag'],
)
summary = run_rag(sc)

### `RAG_MAINTENANCE_04` — 토크가 높을 때 어떤 조치를 해야 해?

**대화 턴:**
1. 토크가 높을 때 어떤 조치를 해야 해?

check: `_check_rag_only` | 태그: maintenance, rag

In [15]:
sc = Scenario(
    sid='RAG_MAINTENANCE_04',
    description='토크가 높을 때 어떤 조치를 해야 해?',
    turns=[
        Turn('토크가 높을 때 어떤 조치를 해야 해?'),
    ],
    check=R._check_rag_only,
    tags=['maintenance', 'rag'],
)
summary = run_rag(sc)

[RAG_MAINTENANCE_04] 토크가 높을 때 어떤 조치를 해야 해?
  PASS ✅ (11.22s)
  tasks=['evidence', 'final_answer']
  유사도 score: top=0.455 평균=0.453 전체=[0.455, 0.451]
    rag_trace: status=OK profile=fallback_broad mode=A
      search_query: 토크가 높을 때 어떤 조치를 해야 해?

[Supervisor evidence focus]
토크가 높을 때의 안전 절차와 조치 방법

[Gate feedback]
failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.
      expansion_tags: []
      priority_docs: ['Mechanical Service', 'Mill Spindle', 'Mill Chatter'] | failure_types: []
      retrieved (source/chunk/score):
        - kosha/M-192-2017 기계안전을 위한 제어시스템의 안전관련부품류 설계 기술지침.pdf (chunk=26, page=None) score=0.455
        - kosha/M-192-2017 기계안전을 위한 제어시스템의 안전관련부품류 설계 기술지침.pdf (chunk=33, page=None) score=0.451
  answer=📄 종합 판단: 점검 문서 근거 요약  토크가 높을 때는 장비의 부하 상태와 윤활 상태를 점검하고, 필요시 관련 문서에 따라 조정이 필요하다.  문서 근거 - 토크가 높을 때는 기계의 작동 방식이 위험한 상황을 초래할 수 있으므로, 적합한 수단을 통해 이를 방지해야 한다. 이는 모드 선택 스위치가 기계 작동을 직접적으로 유발하지 않도록 하고, 운전자가 별도의 작동을 요구하도록 설계되어야 한다

### `RAG_MAINTENANCE_05` — 과열이 발생했을 때 대응 절차 알려줘.

**대화 턴:**
1. 과열이 발생했을 때 대응 절차 알려줘.

check: `_check_rag_only` | 태그: maintenance, rag

In [ ]:
sc = Scenario(
    sid='RAG_MAINTENANCE_05',
    description='과열이 발생했을 때 대응 절차 알려줘.',
    turns=[
        Turn('과열이 발생했을 때 대응 절차 알려줘.'),
    ],
    check=R._check_rag_only,
    tags=['maintenance', 'rag'],
)
summary = run_rag(sc)

### `RAG_MAINTENANCE_06` — 작업 재개 전에 확인해야 하는 사항은?

**대화 턴:**
1. 작업 재개 전에 확인해야 하는 사항은?

check: `_check_rag_only` | 태그: maintenance, rag

In [ ]:
sc = Scenario(
    sid='RAG_MAINTENANCE_06',
    description='작업 재개 전에 확인해야 하는 사항은?',
    turns=[
        Turn('작업 재개 전에 확인해야 하는 사항은?'),
    ],
    check=R._check_rag_only,
    tags=['maintenance', 'rag'],
)
summary = run_rag(sc)

## 예방 (preventive) — 단순 RAG

### `RAG_PREVENTIVE_01` — 공구 수명을 늘리는 방법 알려줘.

**대화 턴:**
1. 공구 수명을 늘리는 방법 알려줘.

check: `_check_rag_only` | 태그: preventive, rag

In [ ]:
sc = Scenario(
    sid='RAG_PREVENTIVE_01',
    description='공구 수명을 늘리는 방법 알려줘.',
    turns=[
        Turn('공구 수명을 늘리는 방법 알려줘.'),
    ],
    check=R._check_rag_only,
    tags=['preventive', 'rag'],
)
summary = run_rag(sc)

### `RAG_PREVENTIVE_02` — 예방 점검 주기는 어떻게 잡아?

**대화 턴:**
1. 예방 점검 주기는 어떻게 잡아?

check: `_check_rag_only` | 태그: preventive, rag

In [ ]:
sc = Scenario(
    sid='RAG_PREVENTIVE_02',
    description='예방 점검 주기는 어떻게 잡아?',
    turns=[
        Turn('예방 점검 주기는 어떻게 잡아?'),
    ],
    check=R._check_rag_only,
    tags=['preventive', 'rag'],
)
summary = run_rag(sc)

### `RAG_PREVENTIVE_03` — 과부하를 방지하려면 어떻게 해야 해?

**대화 턴:**
1. 과부하를 방지하려면 어떻게 해야 해?

check: `_check_rag_only` | 태그: preventive, rag

In [16]:
sc = Scenario(
    sid='RAG_PREVENTIVE_03',
    description='과부하를 방지하려면 어떻게 해야 해?',
    turns=[
        Turn('과부하를 방지하려면 어떻게 해야 해?'),
    ],
    check=R._check_rag_only,
    tags=['preventive', 'rag'],
)
summary = run_rag(sc)

[RAG_PREVENTIVE_03] 과부하를 방지하려면 어떻게 해야 해?
  PASS ✅ (12.28s)
  tasks=['evidence', 'final_answer']
  유사도 score: top=0.468 평균=0.468 전체=[0.468]
    rag_trace: status=OK profile=fallback_broad mode=A
      search_query: 과부하를 방지하려면 어떻게 해야 해?

[Supervisor evidence focus]
과부하 방지 절차, 안전 절차, 예방 조치, failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.

[Gate feedback]
failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.
      expansion_tags: []
      priority_docs: ['Mechanical Service', 'Mill Spindle', 'Mill Chatter'] | failure_types: []
      retrieved (source/chunk/score):
        - kosha/M-131-2012 기계의 결함진단을 위한 자료해석 기술지침.pdf (chunk=5, page=None) score=0.468
  answer=📄 종합 판단: 점검 문서 근거 요약  장비의 과부하를 방지하려면 정기적인 점검과 부하 모니터링 시스템의 설정 확인이 필요합니다.  문서 근거 - 과부하(OSF)를 방지하기 위해서는 기계의 소음, 냄새, 온도, 습도, 누설 등 변화를 사람이 감지하고, 경보 시스템을 통해 데이터를 확인하는 것이 중요하다. 이러한 방법으로 기계의 이상 상태를 조기에 발견할 수 있다[C1]. - 과부하의 원인을 진단하기 위해서는 전

### `RAG_PREVENTIVE_04` — 스핀들 수명을 늘리는 관리 방법 알려줘.

**대화 턴:**
1. 스핀들 수명을 늘리는 관리 방법 알려줘.

check: `_check_rag_only` | 태그: preventive, rag

In [17]:
sc = Scenario(
    sid='RAG_PREVENTIVE_04',
    description='스핀들 수명을 늘리는 관리 방법 알려줘.',
    turns=[
        Turn('스핀들 수명을 늘리는 관리 방법 알려줘.'),
    ],
    check=R._check_rag_only,
    tags=['preventive', 'rag'],
)
summary = run_rag(sc)

[RAG_PREVENTIVE_04] 스핀들 수명을 늘리는 관리 방법 알려줘.
  PASS ✅ (12.92s)
  tasks=['evidence', 'final_answer']
  유사도 score: top=0.453 평균=0.453 전체=[0.453]
    rag_trace: status=OK profile=fallback_broad mode=A
      search_query: 스핀들 수명을 늘리는 관리 방법 알려줘.

[Supervisor evidence focus]
스핀들 수명 관리 방법, 예방 유지보수 절차, 스핀들 관리 매뉴얼, failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.

[Gate feedback]
failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.
      expansion_tags: []
      priority_docs: ['Mechanical Service', 'Mill Spindle', 'Mill Chatter'] | failure_types: []
      retrieved (source/chunk/score):
        - kosha/M-114-2012 윤활유 분석에 의한 고장진단 기술지침.pdf (chunk=4, page=None) score=0.453
  answer=📄 종합 판단: 점검 문서 근거 요약  스핀들 수명을 늘리기 위해서는 정기적인 윤활과 청소, 진동 및 온도 모니터링이 필요하며, 제조사의 유지보수 지침을 따르는 것이 중요하다.  문서 근거 - 스핀들의 수명을 늘리기 위해서는 윤활유의 상태를 지속적으로 감시하고 최적의 교환 시기를 결정하는 것이 중요하다. 윤활유의 품질을 유지하기 위해서는 정기적인 시료 채취와 분석이 필요하며, 이는 기

## 예측 기반 RAG — 단일 턴 (prediction + evidence)

### `RAG_PRED_SINGLE_01` — 입력한 데이터로 고장 위험을 진단하고 관련 정비 기준도 알려줘.

**대화 턴:**
1. 입력한 데이터로 고장 위험을 진단하고 관련 정비 기준도 알려줘. _(+ 입력 피처 FEATURES_HIGH_RISK)_

check: `_check_prediction_rag_single` | 태그: prediction_rag_single, rag

In [18]:
sc = Scenario(
    sid='RAG_PRED_SINGLE_01',
    description='입력한 데이터로 고장 위험을 진단하고 관련 정비 기준도 알려줘.',
    turns=[
        Turn('입력한 데이터로 고장 위험을 진단하고 관련 정비 기준도 알려줘.', FEATURES_HIGH_RISK),
    ],
    check=R._check_prediction_rag_single,
    tags=['prediction_rag_single', 'rag'],
)
summary = run_rag(sc)

[RAG_PRED_SINGLE_01] 입력한 데이터로 고장 위험을 진단하고 관련 정비 기준도 알려줘.
  PASS ✅ (13.08s)
  tasks=['evidence', 'final_answer', 'prediction']
  유사도 score: top=0.578 평균=0.560 전체=[0.578, 0.561, 0.56, 0.542]
    rag_trace: status=OK profile=prediction_plus_rag mode=B
      search_query: 입력한 데이터로 고장 위험을 진단하고 관련 정비 기준도 알려줘.

[Supervisor evidence focus]
정비 기준 과부하 불량 tool load cutting load cutting force heavy cutting heavy milling aggressive feedrate feedrate
      expansion_tags: ['과부하 불량', 'tool load', 'cutting load', 'cutting force', 'heavy cutting', 'heavy milling', 'aggressive feedrate', 'feedrate', 'speeds and feeds', 'depth of cut', 'width of cut', 'tool engagement']
      priority_docs: ['Mill Chatter', 'Mechanical Service', 'Mill Spindle'] | failure_types: ['OSF', 'TWF', 'HDF', 'PWF']
      retrieved (source/chunk/score):
        - haas/haascnc.com-Mill Chatter - Troubleshooting - TG0100.pdf (chunk=2, page=None) score=0.578
        - haas/haascnc.com-Mill Chatter - Troubleshooting - TG0100.pdf (chun

### `RAG_PRED_SINGLE_02` — 입력한 데이터로 HDF 위험을 진단하고 지금 가장 먼저 해야 할 점검을 알려줘.

**대화 턴:**
1. 입력한 데이터로 HDF 위험을 진단하고 지금 가장 먼저 해야 할 점검을 알려줘. _(+ 입력 피처 FEATURES_HIGH_RISK)_

check: `_check_prediction_rag_single` | 태그: prediction_rag_single, rag

In [ ]:
sc = Scenario(
    sid='RAG_PRED_SINGLE_02',
    description='입력한 데이터로 HDF 위험을 진단하고 지금 가장 먼저 해야 할 점검을 알려줘.',
    turns=[
        Turn('입력한 데이터로 HDF 위험을 진단하고 지금 가장 먼저 해야 할 점검을 알려줘.', FEATURES_HIGH_RISK),
    ],
    check=R._check_prediction_rag_single,
    tags=['prediction_rag_single', 'rag'],
)
summary = run_rag(sc)

### `RAG_PRED_SINGLE_03` — 입력한 데이터로 HDF 위험을 진단하고 작업을 계속해도 되는지 문서 근거와 함께 알려줘.

**대화 턴:**
1. 입력한 데이터로 HDF 위험을 진단하고 작업을 계속해도 되는지 문서 근거와 함께 알려줘. _(+ 입력 피처 FEATURES_HIGH_RISK)_

check: `_check_prediction_rag_single` | 태그: prediction_rag_single, rag

In [19]:
sc = Scenario(
    sid='RAG_PRED_SINGLE_03',
    description='입력한 데이터로 HDF 위험을 진단하고 작업을 계속해도 되는지 문서 근거와 함께 알려줘.',
    turns=[
        Turn('입력한 데이터로 HDF 위험을 진단하고 작업을 계속해도 되는지 문서 근거와 함께 알려줘.', FEATURES_HIGH_RISK),
    ],
    check=R._check_prediction_rag_single,
    tags=['prediction_rag_single', 'rag'],
)
summary = run_rag(sc)

[RAG_PRED_SINGLE_03] 입력한 데이터로 HDF 위험을 진단하고 작업을 계속해도 되는지 문서 근거와 함께 알려줘.
  PASS ✅ (14.86s)
  tasks=['evidence', 'final_answer', 'prediction']
  유사도 score: top=0.578 평균=0.560 전체=[0.578, 0.561, 0.56, 0.542]
    rag_trace: status=OK profile=prediction_plus_rag mode=B
      search_query: 입력한 데이터로 HDF 위험을 진단하고 작업을 계속해도 되는지 문서 근거와 함께 알려줘.

[Supervisor evidence focus]
작업 지속 가능성에 대한 문서 근거 과부하 불량 tool load cutting load cutting force heavy cutting heavy milling aggressive feedrate feedrate
      expansion_tags: ['과부하 불량', 'tool load', 'cutting load', 'cutting force', 'heavy cutting', 'heavy milling', 'aggressive feedrate', 'feedrate', 'speeds and feeds', 'depth of cut', 'width of cut', 'tool engagement']
      priority_docs: ['Mill Chatter', 'Mechanical Service', 'Mill Spindle'] | failure_types: ['OSF', 'TWF', 'HDF', 'PWF']
      retrieved (source/chunk/score):
        - haas/haascnc.com-Mill Chatter - Troubleshooting - TG0100.pdf (chunk=2, page=None) score=0.578
        - haas/haascnc.com-Mill Ch

### `RAG_PRED_SINGLE_04` — 입력한 데이터로 HDF 위험을 진단하고 어떤 부품부터 확인해야 하는지 알려줘.

**대화 턴:**
1. 입력한 데이터로 HDF 위험을 진단하고 어떤 부품부터 확인해야 하는지 알려줘. _(+ 입력 피처 FEATURES_HIGH_RISK)_

check: `_check_prediction_rag_single` | 태그: prediction_rag_single, rag

In [ ]:
sc = Scenario(
    sid='RAG_PRED_SINGLE_04',
    description='입력한 데이터로 HDF 위험을 진단하고 어떤 부품부터 확인해야 하는지 알려줘.',
    turns=[
        Turn('입력한 데이터로 HDF 위험을 진단하고 어떤 부품부터 확인해야 하는지 알려줘.', FEATURES_HIGH_RISK),
    ],
    check=R._check_prediction_rag_single,
    tags=['prediction_rag_single', 'rag'],
)
summary = run_rag(sc)

### `RAG_PRED_SINGLE_05` — 입력한 데이터로 HDF 위험을 진단하고 점검 순서를 알려줘.

**대화 턴:**
1. 입력한 데이터로 HDF 위험을 진단하고 점검 순서를 알려줘. _(+ 입력 피처 FEATURES_HIGH_RISK)_

check: `_check_prediction_rag_single` | 태그: prediction_rag_single, rag

In [ ]:
sc = Scenario(
    sid='RAG_PRED_SINGLE_05',
    description='입력한 데이터로 HDF 위험을 진단하고 점검 순서를 알려줘.',
    turns=[
        Turn('입력한 데이터로 HDF 위험을 진단하고 점검 순서를 알려줘.', FEATURES_HIGH_RISK),
    ],
    check=R._check_prediction_rag_single,
    tags=['prediction_rag_single', 'rag'],
)
summary = run_rag(sc)

### `RAG_PRED_SINGLE_06` — 입력한 데이터로 HDF 위험을 진단하고 작업을 중단해야 하는 상황인지 알려줘.

**대화 턴:**
1. 입력한 데이터로 HDF 위험을 진단하고 작업을 중단해야 하는 상황인지 알려줘. _(+ 입력 피처 FEATURES_HIGH_RISK)_

check: `_check_prediction_rag_single` | 태그: prediction_rag_single, rag

In [ ]:
sc = Scenario(
    sid='RAG_PRED_SINGLE_06',
    description='입력한 데이터로 HDF 위험을 진단하고 작업을 중단해야 하는 상황인지 알려줘.',
    turns=[
        Turn('입력한 데이터로 HDF 위험을 진단하고 작업을 중단해야 하는 상황인지 알려줘.', FEATURES_HIGH_RISK),
    ],
    check=R._check_prediction_rag_single,
    tags=['prediction_rag_single', 'rag'],
)
summary = run_rag(sc)

## 예측 기반 RAG — 멀티턴 (1턴 예측 → 2턴 후속 RAG)

### `RAG_PRED_MULTI_01` — 이런 상황에서 관련 정비 기준 있어?

**대화 턴:**
1. 입력한 데이터로 고장 위험을 진단해줘. _(+ 입력 피처 FEATURES_HIGH_RISK)_
2. 이런 상황에서 관련 정비 기준 있어?

check: `_check_prediction_rag_multiturn` | 태그: prediction_rag_multiturn, rag

In [20]:
sc = Scenario(
    sid='RAG_PRED_MULTI_01',
    description='이런 상황에서 관련 정비 기준 있어?',
    turns=[
        Turn('입력한 데이터로 고장 위험을 진단해줘.', FEATURES_HIGH_RISK),
        Turn('이런 상황에서 관련 정비 기준 있어?'),
    ],
    check=R._check_prediction_rag_multiturn,
    tags=['prediction_rag_multiturn', 'rag'],
)
summary = run_rag(sc)

[RAG_PRED_MULTI_01] 이런 상황에서 관련 정비 기준 있어?
  PASS ✅ (19.06s)
  tasks=['evidence', 'final_answer']
  유사도 score: top=0.452 평균=0.452 전체=[0.452]
    rag_trace: status=OK profile=fallback_broad mode=A
      search_query: 이런 상황에서 관련 정비 기준 있어?

[Supervisor evidence focus]
정비 기준, failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.
      expansion_tags: []
      priority_docs: ['Mechanical Service', 'Mill Spindle', 'Mill Chatter'] | failure_types: []
      retrieved (source/chunk/score):
        - kosha/M-131-2012 기계의 결함진단을 위한 자료해석 기술지침.pdf (chunk=5, page=None) score=0.452
  answer=📄 종합 판단: 점검 문서 근거 요약  관련 문서에 근거가 있으니 정비 기준을 확인하고 점검을 진행해야 합니다.  문서 근거 - 기계의 결함 진단을 위해서는 소음, 냄새, 온도, 습도, 누설 등 기계의 변화를 사람이 감지하고, 경보가 울리는 것에 의해 데이터를 확인하는 것이 중요하다. 이러한 증상들을 종합적으로 평가하여 결함 가설을 생성할 수 있다 [C1]. - 결함 가설을 작성할 때는 전체적인 증상과 결함의 관계를 고려하여 목록을 작성하고, 요구되는 증상과 적합하지 않은 가설들을 제거하는 평가 작업을 수행해야 한다 [C1]. - 결함 가설의 확정 단계에서는 과거의 데이터, 같은 형식의 기계, 같은 운전 조건들로부터 얻어진 결함 발생 확률과 결함의 심각도

### `RAG_PRED_MULTI_02` — 지금 가장 먼저 해야 할 점검은?

**대화 턴:**
1. 입력한 데이터로 고장 위험을 진단해줘. _(+ 입력 피처 FEATURES_HIGH_RISK)_
2. 지금 가장 먼저 해야 할 점검은?

check: `_check_prediction_rag_multiturn` | 태그: prediction_rag_multiturn, rag

In [ ]:
sc = Scenario(
    sid='RAG_PRED_MULTI_02',
    description='지금 가장 먼저 해야 할 점검은?',
    turns=[
        Turn('입력한 데이터로 고장 위험을 진단해줘.', FEATURES_HIGH_RISK),
        Turn('지금 가장 먼저 해야 할 점검은?'),
    ],
    check=R._check_prediction_rag_multiturn,
    tags=['prediction_rag_multiturn', 'rag'],
)
summary = run_rag(sc)

### `RAG_PRED_MULTI_03` — 작업 계속해도 돼?

**대화 턴:**
1. 입력한 데이터로 고장 위험을 진단해줘. _(+ 입력 피처 FEATURES_HIGH_RISK)_
2. 작업 계속해도 돼?

check: `_check_prediction_rag_multiturn` | 태그: prediction_rag_multiturn, rag

In [21]:
sc = Scenario(
    sid='RAG_PRED_MULTI_03',
    description='작업 계속해도 돼?',
    turns=[
        Turn('입력한 데이터로 고장 위험을 진단해줘.', FEATURES_HIGH_RISK),
        Turn('작업 계속해도 돼?'),
    ],
    check=R._check_prediction_rag_multiturn,
    tags=['prediction_rag_multiturn', 'rag'],
)
summary = run_rag(sc)

[RAG_PRED_MULTI_03] 작업 계속해도 돼?
  PASS ✅ (21.38s)
  tasks=['evidence', 'final_answer']
  유사도 score: top=0.490 평균=0.484 전체=[0.49, 0.488, 0.483, 0.475]
    rag_trace: status=OK profile=safety_procedure_rag mode=A
      search_query: 작업 계속해도 돼?

[Supervisor evidence focus]
안전 절차, 점검 필요성
      expansion_tags: []
      priority_docs: ['Mechanical Service', 'Mill Spindle', 'Mill Chatter'] | failure_types: []
      retrieved (source/chunk/score):
        - kosha/B-M-25-2026 에너지 차단장치의 잠금·표지에 관한 기술지원규정.pdf (chunk=1, page=None) score=0.49
        - kosha/B-M-25-2026 에너지 차단장치의 잠금·표지에 관한 기술지원규정.pdf (chunk=8, page=None) score=0.488
        - kosha/M-192-2017 기계안전을 위한 제어시스템의 안전관련부품류 설계 기술지침.pdf (chunk=5, page=None) score=0.483
        - kosha/M-192-2017 기계안전을 위한 제어시스템의 안전관련부품류 설계 기술지침.pdf (chunk=33, page=None) score=0.475
  answer=📄 종합 판단: 점검 문서 근거 요약  작업을 계속하기 전에 관련 문서를 검토하여 안전과 절차 준수 여부를 확인해야 합니다.  문서 근거 - 작업을 계속하기 전에 에너지 차단장치의 잠금 및 표지 설치가 적절히 이루어졌는지 확인해야 한다. 이는 작업자의 안전을 보호하기 위한 필수 절차로, 잠금장치와 표지에는 

### `RAG_PRED_MULTI_04` — 어떤 부품부터 확인해야 해?

**대화 턴:**
1. 입력한 데이터로 고장 위험을 진단해줘. _(+ 입력 피처 FEATURES_HIGH_RISK)_
2. 어떤 부품부터 확인해야 해?

check: `_check_prediction_rag_multiturn` | 태그: prediction_rag_multiturn, rag

In [22]:
sc = Scenario(
    sid='RAG_PRED_MULTI_04',
    description='어떤 부품부터 확인해야 해?',
    turns=[
        Turn('입력한 데이터로 고장 위험을 진단해줘.', FEATURES_HIGH_RISK),
        Turn('어떤 부품부터 확인해야 해?'),
    ],
    check=R._check_prediction_rag_multiturn,
    tags=['prediction_rag_multiturn', 'rag'],
)
summary = run_rag(sc)

[RAG_PRED_MULTI_04] 어떤 부품부터 확인해야 해?
  FAIL ❌ (15.55s)
  tasks=['final_answer', 'prediction']
  유사도 score: (검색 문서 없음 / NO_EVIDENCE)
  - 2턴 evidence task 없음: {'final_answer', 'prediction'}
  - citation/문서 근거 섹션 없음
  질의:
    1. 입력한 데이터로 고장 위험을 진단해줘. (+입력 피처)
    2. 어떤 부품부터 확인해야 해?
    rag_trace: (evidence_bundle 없음)
  answer=ℹ️ 종합 판단: 입력 부족 — 정확한 진단을 위해 추가 데이터가 필요합니다.  입력 부족 문제를 해결하기 위해 데이터 입력 시스템과 관련된 부품부터 점검이 필요합니다.  추가로 필요한 입력 현재 입력만으로는 정확한 진단이 어렵습니다. 다음 값을 입력해 주세요: 제품 타입, 공기온도, 공정온도, 회전속도, 토크, 공구 마모  ℹ 보조 진단·조회 결과이며, 실제 조치는 현장 담당자 확인이 필요합니다.
  --- 최종 답변 ---
    ℹ️ 종합 판단: 입력 부족 — 정확한 진단을 위해 추가 데이터가 필요합니다.
    
    입력 부족 문제를 해결하기 위해 데이터 입력 시스템과 관련된 부품부터 점검이 필요합니다.
    
    추가로 필요한 입력
    현재 입력만으로는 정확한 진단이 어렵습니다. 다음 값을 입력해 주세요: 제품 타입, 공기온도, 공정온도, 회전속도, 토크, 공구 마모
    
    ℹ 보조 진단·조회 결과이며, 실제 조치는 현장 담당자 확인이 필요합니다.


### `RAG_PRED_MULTI_05` — 점검 순서를 알려줘.

**대화 턴:**
1. 입력한 데이터로 고장 위험을 진단해줘. _(+ 입력 피처 FEATURES_HIGH_RISK)_
2. 점검 순서를 알려줘.

check: `_check_prediction_rag_multiturn` | 태그: prediction_rag_multiturn, rag

In [23]:
sc = Scenario(
    sid='RAG_PRED_MULTI_05',
    description='점검 순서를 알려줘.',
    turns=[
        Turn('입력한 데이터로 고장 위험을 진단해줘.', FEATURES_HIGH_RISK),
        Turn('점검 순서를 알려줘.'),
    ],
    check=R._check_prediction_rag_multiturn,
    tags=['prediction_rag_multiturn', 'rag'],
)
summary = run_rag(sc)

[RAG_PRED_MULTI_05] 점검 순서를 알려줘.
  PASS ✅ (22.97s)
  tasks=['evidence', 'final_answer']
  유사도 score: top=0.471 평균=0.461 전체=[0.471, 0.451]
    rag_trace: status=OK profile=fallback_broad mode=A
      search_query: 점검 순서를 알려줘.

[Supervisor evidence focus]
점검 순서, 안전 절차

[Gate feedback]
failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.
      expansion_tags: []
      priority_docs: ['Mechanical Service', 'Mill Spindle', 'Mill Chatter'] | failure_types: []
      retrieved (source/chunk/score):
        - kosha/M-192-2017 기계안전을 위한 제어시스템의 안전관련부품류 설계 기술지침.pdf (chunk=33, page=None) score=0.471
        - kosha/B-M-25-2026 에너지 차단장치의 잠금·표지에 관한 기술지원규정.pdf (chunk=8, page=None) score=0.451
  answer=📄 종합 판단: 점검 문서 근거 요약  관련 문서를 확인하여 점검 순서를 검토하고, 필요한 절차를 따라 점검을 진행하세요.  문서 근거 - 안전기능의 점검은 기계 시동 시, 위험한 상황의 개시 전, 새로운 주기의 시작 등에서 주기적으로 수행되어야 한다. 이러한 점검은 자동으로 시작될 수 있으며, 결함이 검출되지 않으면 작동을 허가하고, 결함이 검출되면 적절한 제어행동을 촉발하는 출력을 발생시켜야 한다[C1]. - 에너지 차단장치의 잠금 및 표지에 관한 정

### `RAG_PRED_MULTI_06` — 작업을 중단해야 하는 상황이야?

**대화 턴:**
1. 입력한 데이터로 고장 위험을 진단해줘. _(+ 입력 피처 FEATURES_HIGH_RISK)_
2. 작업을 중단해야 하는 상황이야?

check: `_check_prediction_rag_multiturn` | 태그: prediction_rag_multiturn, rag

In [ ]:
sc = Scenario(
    sid='RAG_PRED_MULTI_06',
    description='작업을 중단해야 하는 상황이야?',
    turns=[
        Turn('입력한 데이터로 고장 위험을 진단해줘.', FEATURES_HIGH_RISK),
        Turn('작업을 중단해야 하는 상황이야?'),
    ],
    check=R._check_prediction_rag_multiturn,
    tags=['prediction_rag_multiturn', 'rag'],
)
summary = run_rag(sc)

### `RAG_PRED_MULTI_07` — 토크가 높으면 왜 OSF가 발생하지?

**대화 턴:**
1. 입력한 데이터로 고장 위험을 진단해줘. _(+ 입력 피처 FEATURES_HIGH_RISK)_
2. 토크가 높으면 왜 OSF가 발생하지?

check: `_check_prediction_rag_multiturn` | 태그: prediction_rag_multiturn, rag

In [ ]:
sc = Scenario(
    sid='RAG_PRED_MULTI_07',
    description='토크가 높으면 왜 OSF가 발생하지?',
    turns=[
        Turn('입력한 데이터로 고장 위험을 진단해줘.', FEATURES_HIGH_RISK),
        Turn('토크가 높으면 왜 OSF가 발생하지?'),
    ],
    check=R._check_prediction_rag_multiturn,
    tags=['prediction_rag_multiturn', 'rag'],
)
summary = run_rag(sc)

### `RAG_PRED_MULTI_10` — 공구를 교체하면 괜찮을까?

**대화 턴:**
1. 입력한 데이터로 고장 위험을 진단해줘. _(+ 입력 피처 FEATURES_HIGH_RISK)_
2. 공구를 교체하면 괜찮을까?

check: `_check_prediction_rag_multiturn` | 태그: prediction_rag_multiturn, rag

In [ ]:
sc = Scenario(
    sid='RAG_PRED_MULTI_10',
    description='공구를 교체하면 괜찮을까?',
    turns=[
        Turn('입력한 데이터로 고장 위험을 진단해줘.', FEATURES_HIGH_RISK),
        Turn('공구를 교체하면 괜찮을까?'),
    ],
    check=R._check_prediction_rag_multiturn,
    tags=['prediction_rag_multiturn', 'rag'],
)
summary = run_rag(sc)

## 근거 없음 (empty) — NO_EVIDENCE fallback

### `RAG_EMPTY_01` — 용접 로봇 토치 케이블 교체 주기에 대한 정비 문서 근거를 찾아줘.

**대화 턴:**
1. 용접 로봇 토치 케이블 교체 주기에 대한 정비 문서 근거를 찾아줘.

check: `_check_no_evidence_fallback` | 태그: empty, rag

In [25]:
sc = Scenario(
    sid='RAG_EMPTY_01',
    description='용접 로봇 토치 케이블 교체 주기에 대한 정비 문서 근거를 찾아줘.',
    turns=[
        Turn('용접 로봇 토치 케이블 교체 주기에 대한 정비 문서 근거를 찾아줘.'),
    ],
    check=R._check_no_evidence_fallback,
    tags=['empty', 'rag'],
)
summary = run_rag(sc)

[RAG_EMPTY_01] 용접 로봇 토치 케이블 교체 주기에 대한 정비 문서 근거를 찾아줘.
  PASS ✅ (13.65s)
  tasks=['evidence', 'final_answer']
  유사도 score: top=0.452 평균=0.452 전체=[0.452]
    rag_trace: status=OK profile=fallback_broad mode=A
      search_query: 용접 로봇 토치 케이블 교체 주기에 대한 정비 문서 근거를 찾아줘.

[Supervisor evidence focus]
용접 로봇 토치 케이블 교체 주기, 정비 문서 근거, failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요., failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.

[Gate feedback]
failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.
      expansion_tags: []
      priority_docs: ['Mechanical Service', 'Mill Spindle', 'Mill Chatter'] | failure_types: []
      retrieved (source/chunk/score):
        - kosha/M-131-2012 기계의 결함진단을 위한 자료해석 기술지침.pdf (chunk=2, page=None) score=0.452
  answer=📄 종합 판단: 점검 문서 근거 요약  용접 로봇 토치 케이블의 교체 주기에 대한 정비 문서가 존재하니, 해당 문서를 확인하여 교체 주기를 검토하세요.  문서 근거 

### `RAG_EMPTY_02` — 반도체 노광 장비 렌즈 세정 주기에 대한 Haas 문서 근거를 찾아줘.

**대화 턴:**
1. 반도체 노광 장비 렌즈 세정 주기에 대한 Haas 문서 근거를 찾아줘.

check: `_check_no_evidence_fallback` | 태그: empty, rag

In [24]:
sc = Scenario(
    sid='RAG_EMPTY_02',
    description='반도체 노광 장비 렌즈 세정 주기에 대한 Haas 문서 근거를 찾아줘.',
    turns=[
        Turn('반도체 노광 장비 렌즈 세정 주기에 대한 Haas 문서 근거를 찾아줘.'),
    ],
    check=R._check_no_evidence_fallback,
    tags=['empty', 'rag'],
)
summary = run_rag(sc)

[RAG_EMPTY_02] 반도체 노광 장비 렌즈 세정 주기에 대한 Haas 문서 근거를 찾아줘.
  PASS ✅ (12.28s)
  tasks=['evidence', 'final_answer']
  유사도 score: top=0.457 평균=0.457 전체=[0.457, 0.457]
    rag_trace: status=OK profile=fallback_broad mode=A
      search_query: 반도체 노광 장비 렌즈 세정 주기에 대한 Haas 문서 근거를 찾아줘.

[Supervisor evidence focus]
Haas 문서, 렌즈 세정 주기

[Gate feedback]
failure_type, component, symptom, root_cause, corrective/preventive action, 재발 방지 절차 중심으로 검색 질의를 확장하세요.
      expansion_tags: []
      priority_docs: ['Mechanical Service', 'Mill Spindle', 'Mill Chatter'] | failure_types: []
      retrieved (source/chunk/score):
        - haas/english---mechanical-service-manual---2007.pdf (chunk=1, page=None) score=0.457
        - kosha/M-131-2012 기계의 결함진단을 위한 자료해석 기술지침.pdf (chunk=0, page=None) score=0.457
  answer=📄 종합 판단: 점검 문서 근거 요약  렌즈 세정 주기에 대한 Haas 문서의 근거를 확인할 필요가 있습니다.  문서 근거 - Haas 문서에 따르면, 렌즈 세정 주기는 장비의 사용 빈도와 환경 조건에 따라 달라질 수 있으며, 정기적인 점검과 유지보수가 필요하다[C1]. 이는 장비의 성능을 최적화하고 고장을 예방하기 위한 기본적인 절차로 강조된다. - 렌즈 세정 주기를

## 실행 요약

In [ ]:
# 지금까지 이 세션에서 실행한 RAG 시나리오의 PASS/FAIL + 소요 시간 집계
if not RESULTS:
    print("아직 실행한 시나리오가 없습니다.")
else:
    passed = sum(1 for ok in RESULTS.values() if ok)
    total = sum(TIMINGS.values())
    print(f"{passed}/{len(RESULTS)} passed | 총 {total:.2f}s")
    for sid, ok in RESULTS.items():
        print(f"  {'✅' if ok else '❌'} {sid:<28} {TIMINGS.get(sid, 0):6.2f}s")
